In [70]:
from pathlib import Path
import pickle
import torch
import os
from chggen.pl_data.dataset import CHGNetDataset
from chggen.pl_data.datamodule import CrystDataModule
from chggen.pl_modules.model import CHGGen
from chggen.common.data_utils import get_scaler_from_data_list
from torch.utils.data import DataLoader, Dataset
from torch.utils.data.sampler import SubsetRandomSampler
from torch_geometric.data import Batch
import numpy as np
import pytorch_lightning as pl

from types import SimpleNamespace

from pymatgen.core import Structure, Lattice, Species, Element
from pymatgen.core.periodic_table import Element



def get_scaler(dataset, use_prop_scaler = False, 
               scaler_path = None):
    # Load once to compute property scaler
    if scaler_path is None:
        lattice_scaler = get_scaler_from_data_list(
            dataset.cached_data,
            key='scaled_lattice')
        if use_prop_scaler:
            NotImplementedError("Not implemented the multi prop scaler yet.")
    else:
        lattice_scaler = torch.load(
            Path(scaler_path) / 'lattice_scaler.pt')
    return lattice_scaler



In [ ]:
dataset = CHGNetDataset(path= '/home/zhongpc/chggen/data/mptrj/MPtrj_debug.csv',
                        name = 'mptrj_debug',
                        prop_list = ['e_hull'],
                        )

lattice_scaler = get_scaler(dataset= dataset)


In [23]:
checkpoint = torch.load('./test_models/chggen.ckpt')


In [24]:
chggen = CHGGen(lattice_scaler= lattice_scaler, 
                hparams_dict= model_hparams)

/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/torch/jit/_check.py:172: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn("The TorchScript type system doesn't support "


CHGNet initialized with 400,438 parameters
CHGNet initialized with 400,438 parameters


In [47]:
new_model = CHGGen.load_from_checkpoint(checkpoint_path="./test_models/trainer.ckpt")
new_model.lattice_scaler = lattice_scaler

CHGNet initialized with 400,438 parameters
CHGNet initialized with 400,438 parameters


/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/torch/jit/_check.py:172: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn("The TorchScript type system doesn't support "


In [56]:
ld_kwargs = SimpleNamespace(n_step_each = 10,
                            step_lr = 1e-4,
                            min_sigma = 0,
                            save_traj = False,
                            disable_bar = False)


# new_model.langevin_dynamics()

In [113]:
z = torch.rand(7, 64)

results = new_model.langevin_dynamics(z = z, ld_kwargs= ld_kwargs)

/home/zhongpc/chggen/chggen/common/data_utils.py:625: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float)
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:33<00:00,  1.47it/s]


In [114]:
lengths = results['lengths']
angles= results['angles']
num_atoms = results['num_atoms']
frac_coords = results['frac_coords']
atom_types = results['atom_types']

In [115]:
batch = torch.arange(len(num_atoms))
batch = batch.repeat_interleave(num_atoms)

In [118]:
for ii in range(len(num_atoms)):
    indices = torch.where(batch == ii)[0]
    
    Latt = Lattice.from_parameters(a = lengths[ii,0], b = lengths[ii,1], c = lengths[ii,2],
                                   alpha= angles[ii, 0], beta= angles[ii,1], gamma=angles[ii, 2])
                                   
    frac_ = frac_coords[indices]
    type_ = atom_types[indices]
    species_ = [Element.from_Z(ele_Z+1) for ele_Z in type_]
    
    s_gen = Structure(lattice= Latt , species= species_, coords= frac_,
                      to_unit_cell=False,coords_are_cartesian=False);
    print(s_gen.composition)
    s_gen.to(filename= './test_models/structures/test_' + str(ii) + '.cif')
    
    
    

F3 Ge1
F12 Ne2
Ge1 F1 Ne1 Yb1
F12 Ne1 Np1
F2 Ge3 Ne2 In1
F1 Cl1 Np1 Si1
F12 Ne2


/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pymatgen/core/periodic_table.py:221: UserWarning: No Pauling electronegativity for Ne. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  warnings.warn(


In [117]:
Latt

Lattice
    abc : 6.265055385715076 7.132294138903374 8.400997161865234
 angles : 90.61877667632461 87.68540163488167 90.33382438515171
 volume : 375.0583871346305
      A : 6.259943962097168 0.0 0.2530228793621063
      B : -0.03847552463412285 7.131774425506592 -0.07702507078647614
      C : 0.0 0.0 8.400997161865234
    pbc : True True True

In [108]:
angles

tensor([[5.6408, 6.0600, 8.0061],
        [6.3545, 6.8401, 9.0305]])

In [75]:
# type_index = 0

# for ii in range(0, len(frac_coords)):
#     print(ii)
    
#     num_ = num_atoms[ii]
#     atom_type = atom_types[type_index:(type_index+num_)]
    
#     type_index += num_
    
    
#     FracCoords =  frac_coords[type_index:(type_index+num_), :]
#     Species = [Element.from_Z(ele_Z) for ele_Z in atom_type]; 

#     Latt = Lattice.from_parameters(a = lengths[ii,0], b = lengths[ii,1], c = lengths[ii,2],
#                                    alpha= angles[ii, 0], beta= angles[ii,1], gamma=angles[ii, 2])
    
#     s_gen = Structure(lattice= Latt , species= Species, coords= FracCoords,
#                       to_unit_cell=False,coords_are_cartesian=False);
#     print(s_gen.composition)
#     s_gen.to(filename= 'test_' + str(ii) + '.cif')
    

In [73]:
Species

[Element O, Element O, Element F, Element O]

In [74]:
FracCoords

tensor([], size=(0, 3))